# Working with Datasets and Data Loaders in Deep Learning

This notebook provides a comprehensive guide to working with datasets and data loaders in deep learning, covering both PyTorch and TensorFlow implementations. We'll explore how to create, load, preprocess, and efficiently use datasets for training neural networks.

## 1. Import Required Libraries
Before diving into datasets and data loaders, let's import the necessary libraries.

In [ ]:
# Core data science and machine learning libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch and related libraries
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms as transforms

# TensorFlow and related libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Utility libraries
import os
import random
from PIL import Image
import requests
from io import BytesIO
from IPython.display import display, Image as IPImage

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)
torch.manual_seed(42)
tf.random.set_seed(42)

print("Libraries imported successfully!")

## 2. Understanding Dataset Concepts

In deep learning, a dataset is a collection of examples that are used to train and evaluate models. Let's explore the fundamental dataset concepts:

### Key Dataset Concepts

1. **Training, Validation, and Test Sets**:
   - Training set: Used to fit the model parameters (weights and biases)
   - Validation set: Used to tune hyperparameters and evaluate model performance during training
   - Test set: Used to evaluate the final model performance

2. **Features and Labels**:
   - Features (x): Input data used by the model to make predictions
   - Labels (y): Target values the model tries to predict

3. **Batch Size**:
   - Number of examples processed in one forward/backward pass
   - Smaller batches require less memory but may lead to less stable gradient estimates
   - Larger batches provide more stable gradient estimates but require more memory

4. **Dataset Formats**:
   - Structured data: CSV, Excel, databases
   - Images: JPG, PNG, TIF
   - Text: TXT, JSON
   - Audio: WAV, MP3
   - Video: MP4, AVI
   - Specialized formats: TFRecord (TensorFlow), HDF5

## 3. Creating Custom Datasets

Let's learn how to create custom datasets for both PyTorch and TensorFlow.

### 3.1 Creating a Custom Dataset in PyTorch

In PyTorch, we create custom datasets by extending the `torch.utils.data.Dataset` class and implementing three required methods:
- `__init__`: Initialize the dataset
- `__len__`: Return the size of the dataset
- `__getitem__`: Return a sample from the dataset at the given index

In [ ]:
# Example: Creating a custom dataset for synthetic data in PyTorch
class SyntheticDataset(Dataset):
    def __init__(self, num_samples=1000, feature_dim=10, transform=None):
        # Generate random features and labels for demonstration
        self.features = torch.randn(num_samples, feature_dim)
        self.labels = torch.randint(0, 2, (num_samples,))
        self.transform = transform
        
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        
        if self.transform:
            x = self.transform(x)
            
        return x, y

# Create a synthetic dataset
synthetic_dataset = SyntheticDataset()
print(f"Dataset size: {len(synthetic_dataset)}")
print(f"First sample features shape: {synthetic_dataset[0][0].shape}")
print(f"First sample label: {synthetic_dataset[0][1]}")

# Using the dataset with DataLoader
synthetic_loader = DataLoader(
    synthetic_dataset, 
    batch_size=32, 
    shuffle=True
)

# Get a batch of data
features_batch, labels_batch = next(iter(synthetic_loader))
print(f"Batch shape: {features_batch.shape}, Labels shape: {labels_batch.shape}")

### 3.2 Creating an Image Dataset in PyTorch

Let's create a custom dataset for loading images from a directory.

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# This is just an example and won't run without actual image paths
# image_paths = ['path/to/img1.jpg', 'path/to/img2.jpg', ...]
# labels = [0, 1, ...]
# transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])
# image_dataset = ImageDataset(image_paths, labels, transform)

### 3.3 Creating Custom Datasets in TensorFlow

TensorFlow provides a high-level API for working with data: `tf.data.Dataset`. Let's create custom datasets using this API.

In [ ]:
# Example: Creating a synthetic dataset in TensorFlow
def create_tf_synthetic_dataset(num_samples=1000, feature_dim=10, batch_size=32):
    # Generate random features and labels
    features = tf.random.normal((num_samples, feature_dim))
    labels = tf.random.uniform((num_samples,), 0, 2, dtype=tf.int32)
    
    # Create a TensorFlow dataset
    dataset = tf.data.Dataset.from_tensor_slices((features, labels))
    
    # Shuffle and batch the dataset
    dataset = dataset.shuffle(buffer_size=num_samples)
    dataset = dataset.batch(batch_size)
    
    return dataset

# Create a TensorFlow dataset
tf_synthetic_dataset = create_tf_synthetic_dataset()

# Inspect the dataset
for features, labels in tf_synthetic_dataset.take(1):
    print(f"TensorFlow batch features shape: {features.shape}")
    print(f"TensorFlow batch labels shape: {labels.shape}")

## 4. Using Built-in Datasets

Both PyTorch and TensorFlow provide convenient access to popular datasets. Let's explore how to use these built-in datasets.

### 4.1 Built-in Datasets in PyTorch (torchvision)

PyTorch provides many popular datasets through the `torchvision.datasets` module. Let's load the MNIST dataset as an example.

In [ ]:
# Define transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load MNIST dataset
train_dataset = torchvision.datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform
)

print(f"MNIST training set size: {len(train_dataset)}")
print(f"MNIST test set size: {len(test_dataset)}")

# Visualize a sample from the MNIST dataset
image, label = train_dataset[0]
plt.figure(figsize=(2, 2))
plt.imshow(image.squeeze().numpy(), cmap='gray')
plt.title(f"Label: {label}")
plt.axis('off')
plt.show()

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

### 4.2 Other Popular Datasets in torchvision

`torchvision.datasets` provides many other popular datasets:
- CIFAR-10 and CIFAR-100
- ImageNet
- COCO
- Pascal VOC
- FashionMNIST
- STL10

In [ ]:
# Example: Loading CIFAR-10 dataset
cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

cifar_train = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, 
    download=True, 
    transform=cifar_transform
)

# Visualize some samples from CIFAR-10
plt.figure(figsize=(10, 2))
for i in range(5):
    img, label = cifar_train[i]
    plt.subplot(1, 5, i+1)
    plt.imshow(img.permute(1, 2, 0).numpy() * 0.5 + 0.5)  # Denormalize
    plt.title(cifar_train.classes[label])
    plt.axis('off')
plt.tight_layout()
plt.show()

### 4.3 Built-in Datasets in TensorFlow (keras.datasets)

TensorFlow provides popular datasets through the `keras.datasets` module. Let's load the MNIST dataset as an example.

In [ ]:
# Load MNIST dataset from TensorFlow
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Normalize pixel values to be between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0

print(f"TensorFlow MNIST training set shape: {x_train.shape}")
print(f"TensorFlow MNIST test set shape: {x_test.shape}")

# Visualize a sample from the MNIST dataset
plt.figure(figsize=(2, 2))
plt.imshow(x_train[0], cmap='gray')
plt.title(f"Label: {y_train[0]}")
plt.axis('off')
plt.show()

# Create TensorFlow datasets
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(64)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(64)

## 5. Introduction to Data Loaders

Data loaders help us efficiently load and preprocess data in batches. Both PyTorch and TensorFlow have their own implementations of data loaders.

### 5.1 PyTorch DataLoader

PyTorch's `DataLoader` class provides an iterator over a dataset, handling batching, shuffling, and parallel data loading.

Key parameters:
- `dataset`: The dataset from which to load data
- `batch_size`: How many samples per batch to load
- `shuffle`: Whether to shuffle the data at every epoch
- `num_workers`: How many subprocesses to use for data loading
- `drop_last`: Drop the last incomplete batch
- `collate_fn`: Custom function to merge samples into batches

In [ ]:
# Create a DataLoader for the synthetic dataset
synthetic_loader = DataLoader(
    synthetic_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,  # Set to higher values for parallel loading
    drop_last=False,
    pin_memory=True  # This helps speed up data transfer to GPU
)

# Iterate through the DataLoader
for batch_idx, (data, target) in enumerate(synthetic_loader):
    print(f"Batch {batch_idx}: data shape {data.shape}, target shape {target.shape}")
    if batch_idx == 2:  # Print only first few batches
        break

### 5.2 TensorFlow Data API

TensorFlow's `tf.data.Dataset` provides methods for efficiently loading and preprocessing data.

Key methods:
- `batch()`: Combines consecutive elements into batches
- `shuffle()`: Randomly shuffles elements
- `map()`: Applies a function to each element
- `cache()`: Caches elements in memory
- `prefetch()`: Prefetches elements for efficient loading

In [ ]:
# Create a TensorFlow dataset and configure it
tf_dataset = tf.data.Dataset.from_tensor_slices((
    np.random.randn(1000, 10).astype(np.float32),
    np.random.randint(0, 2, size=(1000,))
))

tf_dataset = tf_dataset.shuffle(buffer_size=1000)
tf_dataset = tf_dataset.batch(64)
tf_dataset = tf_dataset.prefetch(tf.data.experimental.AUTOTUNE)

# Iterate through the dataset
for batch_idx, (data, target) in enumerate(tf_dataset):
    print(f"Batch {batch_idx}: data shape {data.shape}, target shape {target.shape}")
    if batch_idx == 2:  # Print only first few batches
        break

## 6. Data Preprocessing Techniques

Proper data preprocessing is crucial for effective model training. Let's explore common preprocessing techniques.

### 6.1 Normalization and Standardization

Normalization scales values to a specific range (usually [0,1]), while standardization transforms data to have zero mean and unit variance.

In [ ]:
# Generate example data
data = np.random.normal(loc=5, scale=2, size=(1000, 1))

# Normalization (Min-Max Scaling)
def normalize(data):
    return (data - data.min()) / (data.max() - data.min())

# Standardization (Z-score normalization)
def standardize(data):
    return (data - data.mean()) / data.std()

# Apply both techniques
data_normalized = normalize(data)
data_standardized = standardize(data)

# Visualize the results
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.hist(data, bins=30)
plt.title(f"Original Data\nMean: {data.mean():.2f}, Std: {data.std():.2f}")

plt.subplot(1, 3, 2)
plt.hist(data_normalized, bins=30)
plt.title(f"Normalized Data\nMean: {data_normalized.mean():.2f}, Std: {data_normalized.std():.2f}")

plt.subplot(1, 3, 3)
plt.hist(data_standardized, bins=30)
plt.title(f"Standardized Data\nMean: {data_standardized.mean():.2f}, Std: {data_standardized.std():.2f}")

plt.tight_layout()
plt.show()

# PyTorch transforms for normalization
normalize_transform = transforms.Normalize(mean=[0.5], std=[0.5])

# TensorFlow normalization
def tf_normalize(image, label):
    return tf.cast(image, tf.float32) / 255.0, label

### 6.2 One-Hot Encoding

One-hot encoding converts categorical variables into a form that could be provided to ML algorithms.

In [ ]:
# Example: One-hot encoding in NumPy
labels = np.array([0, 1, 2, 1, 0, 3])
num_classes = 4

# Method 1: Using NumPy
onehot_labels = np.eye(num_classes)[labels]
print("One-hot encoded labels (NumPy):")
print(onehot_labels)

# Method 2: Using TensorFlow
tf_onehot = tf.one_hot(labels, depth=num_classes)
print("\nOne-hot encoded labels (TensorFlow):")
print(tf_onehot.numpy())

# Method 3: Using PyTorch
torch_onehot = torch.nn.functional.one_hot(torch.tensor(labels), num_classes=num_classes)
print("\nOne-hot encoded labels (PyTorch):")
print(torch_onehot.numpy())

### 6.3 Handling Missing Values

Dealing with missing values is an essential preprocessing step for structured data.

In [ ]:
# Create a DataFrame with missing values
df = pd.DataFrame({
    'A': [1, 2, np.nan, 4, 5],
    'B': [np.nan, 2, 3, 4, 5],
    'C': [1, 2, 3, np.nan, 5]
})

print("Original DataFrame:")
print(df)

# 1. Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# 2. Drop rows with missing values
df_dropped = df.dropna()
print("\nAfter dropping rows with NaN:")
print(df_dropped)

# 3. Fill missing values with mean of columns
df_filled_mean = df.fillna(df.mean())
print("\nAfter filling with column means:")
print(df_filled_mean)

# 4. Fill missing values with median
df_filled_median = df.fillna(df.median())
print("\nAfter filling with column medians:")
print(df_filled_median)

# 5. Forward fill (propagate last valid observation forward)
df_ffill = df.fillna(method='ffill')
print("\nAfter forward fill:")
print(df_ffill)

# Converting to PyTorch tensor (after handling NaN values)
torch_tensor = torch.tensor(df_filled_mean.values, dtype=torch.float32)
print("\nPyTorch tensor:")
print(torch_tensor)

## 7. Data Augmentation

Data augmentation is a technique to artificially increase the size of a dataset by creating modified versions of existing data. This helps improve model generalization and reduce overfitting.

### 7.1 Image Data Augmentation in PyTorch

In [ ]:
# PyTorch image augmentation using torchvision.transforms
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Let's load a sample image and apply augmentations
try:
    # Try to download a sample image
    url = "https://upload.wikimedia.org/wikipedia/commons/3/38/Adorable-animal-cat-20787.jpg"
    response = requests.get(url)
    img = Image.open(BytesIO(response.content))
    
    # Display the original image and several augmented versions
    plt.figure(figsize=(15, 3))
    plt.subplot(1, 5, 1)
    plt.imshow(img)
    plt.title('Original')
    plt.axis('off')
    
    for i in range(4):
        # Apply transformations (except normalization for display)
        aug_img = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1)
        ])(img)
        
        plt.subplot(1, 5, i+2)
        plt.imshow(aug_img)
        plt.title(f'Augmentation {i+1}')
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not load or process image: {e}")
    print("Image augmentation example shown without actual images.")

### 7.2 Image Data Augmentation in TensorFlow

In [ ]:
# TensorFlow image augmentation using ImageDataGenerator
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Let's apply augmentation to an image from CIFAR-10
try:
    # Get a sample image from CIFAR-10
    (x_train, _), _ = tf.keras.datasets.cifar10.load_data()
    sample_image = x_train[0]
    
    # Reshape image for ImageDataGenerator (needs batch dimension)
    sample_image_expanded = sample_image.reshape((1,) + sample_image.shape)
    
    # Display original and augmented images
    plt.figure(figsize=(15, 3))
    plt.subplot(1, 5, 1)
    plt.imshow(sample_image)
    plt.title('Original')
    plt.axis('off')
    
    # Generate 4 augmented images
    aug_iter = datagen.flow(sample_image_expanded, batch_size=1)
    for i in range(4):
        aug_img = next(aug_iter)[0].astype('uint8')
        plt.subplot(1, 5, i+2)
        plt.imshow(aug_img)
        plt.title(f'Augmentation {i+1}')
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Could not load or process CIFAR-10 image: {e}")
    print("TensorFlow image augmentation example shown without actual images.")

### 7.3 Text Data Augmentation

Text data can also benefit from augmentation techniques like synonym replacement, random insertion, and random swapping.

In [ ]:
# Simple text augmentation examples

def synonym_replacement(text, n=1):
    """Replace n words in the text with synonyms (simplified example)."""
    words = text.split()
    # This is a simplified example - in real applications, use a synonym dictionary
    synonym_map = {
        'happy': ['joyful', 'content', 'pleased'],
        'sad': ['unhappy', 'melancholy', 'gloomy'],
        'good': ['great', 'excellent', 'fine'],
        'bad': ['poor', 'terrible', 'awful']
    }
    
    for _ in range(min(n, len(words))):
        idx = random.randint(0, len(words) - 1)
        word = words[idx].lower()
        if word in synonym_map:
            words[idx] = random.choice(synonym_map[word])
    
    return ' '.join(words)

def random_swap(text, n=1):
    """Randomly swap n pairs of words in the text."""
    words = text.split()
    if len(words) < 2:
        return text
    
    for _ in range(n):
        idx1, idx2 = random.sample(range(len(words)), 2)
        words[idx1], words[idx2] = words[idx2], words[idx1]
    
    return ' '.join(words)

# Example
text = "This is a good example of text augmentation techniques"
print(f"Original: {text}")
print(f"Synonym replacement: {synonym_replacement(text, n=1)}")
print(f"Random swap: {random_swap(text, n=1)}")

## 8. Handling Imbalanced Datasets

Imbalanced datasets can bias model training. Let's explore techniques to address this issue.

In [ ]:
# Create an imbalanced synthetic dataset
np.random.seed(42)
n_samples = 1000
n_class_1 = 900  # Majority class
n_class_2 = 100  # Minority class

# Features and labels for imbalanced dataset
X_imbalanced = np.vstack([
    np.random.randn(n_class_1, 2),  # Class 0 features
    np.random.randn(n_class_2, 2) + np.array([2, 2])  # Class 1 features
])

y_imbalanced = np.hstack([
    np.zeros(n_class_1),  # Class 0 labels
    np.ones(n_class_2)    # Class 1 labels
])

# Visualize the imbalanced dataset
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(X_imbalanced[:, 0], X_imbalanced[:, 1], c=y_imbalanced, cmap='viridis', alpha=0.6)
plt.title(f"Imbalanced Dataset\nClass 0: {n_class_1}, Class 1: {n_class_2}")
plt.colorbar()

print("Class distribution:")
print(f"Class 0: {np.sum(y_imbalanced == 0)} samples ({np.mean(y_imbalanced == 0)*100:.1f}%)")
print(f"Class 1: {np.sum(y_imbalanced == 1)} samples ({np.mean(y_imbalanced == 1)*100:.1f}%)")

### 8.1 Resampling Techniques

Let's implement oversampling and undersampling to balance the dataset.

In [ ]:
# 1. Random Undersampling
def random_undersample(X, y, ratio=1.0):
    """Randomly undersample the majority class"""
    # Identify classes and their indices
    classes, counts = np.unique(y, return_counts=True)
    minority_class = classes[np.argmin(counts)]
    majority_class = classes[np.argmax(counts)]
    
    # Get indices for each class
    minority_indices = np.where(y == minority_class)[0]
    majority_indices = np.where(y == majority_class)[0]
    
    # Calculate how many samples to keep from majority class
    n_minority = len(minority_indices)
    n_majority_to_keep = int(n_minority * ratio)
    
    # Randomly select samples from majority class
    majority_indices_to_keep = np.random.choice(
        majority_indices, 
        size=n_majority_to_keep, 
        replace=False
    )
    
    # Combine minority and selected majority indices
    indices_to_keep = np.concatenate([minority_indices, majority_indices_to_keep])
    
    # Return balanced dataset
    return X[indices_to_keep], y[indices_to_keep]

# 2. Random Oversampling
def random_oversample(X, y, ratio=1.0):
    """Randomly oversample the minority class"""
    # Identify classes and their indices
    classes, counts = np.unique(y, return_counts=True)
    minority_class = classes[np.argmin(counts)]
    majority_class = classes[np.argmax(counts)]
    
    # Get indices for each class
    minority_indices = np.where(y == minority_class)[0]
    majority_indices = np.where(y == majority_class)[0]
    
    # Calculate how many samples to create for minority class
    n_majority = len(majority_indices)
    n_minority = len(minority_indices)
    n_minority_to_add = int(n_majority * ratio) - n_minority
    
    # Randomly select samples from minority class (with replacement)
    minority_indices_to_add = np.random.choice(
        minority_indices, 
        size=n_minority_to_add, 
        replace=True
    )
    
    # Combine original dataset with oversampled minority
    X_oversampled = np.vstack([X, X[minority_indices_to_add]])
    y_oversampled = np.hstack([y, y[minority_indices_to_add]])
    
    # Return balanced dataset
    return X_oversampled, y_oversampled

# Apply undersampling and oversampling
X_undersampled, y_undersampled = random_undersample(X_imbalanced, y_imbalanced)
X_oversampled, y_oversampled = random_oversample(X_imbalanced, y_imbalanced)

# Display results
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.scatter(X_imbalanced[:, 0], X_imbalanced[:, 1], c=y_imbalanced, cmap='viridis', alpha=0.6)
plt.title(f"Original\nClass 0: {np.sum(y_imbalanced == 0)}, Class 1: {np.sum(y_imbalanced == 1)}")
plt.colorbar()

plt.subplot(1, 3, 2)
plt.scatter(X_undersampled[:, 0], X_undersampled[:, 1], c=y_undersampled, cmap='viridis', alpha=0.6)
plt.title(f"Undersampled\nClass 0: {np.sum(y_undersampled == 0)}, Class 1: {np.sum(y_undersampled == 1)}")
plt.colorbar()

plt.subplot(1, 3, 3)
plt.scatter(X_oversampled[:, 0], X_oversampled[:, 1], c=y_oversampled, cmap='viridis', alpha=0.6)
plt.title(f"Oversampled\nClass 0: {np.sum(y_oversampled == 0)}, Class 1: {np.sum(y_oversampled == 1)}")
plt.colorbar()

plt.tight_layout()
plt.show()

### 8.2 Using Weighted Sampling in Data Loaders

Both PyTorch and TensorFlow support weighted sampling to handle imbalanced datasets during training.

In [ ]:
# Weighted sampling in PyTorch
from torch.utils.data.sampler import WeightedRandomSampler

# Create a simple dataset with the imbalanced data
class ImbalancedDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
        
    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

# Create the dataset
imbalanced_dataset = ImbalancedDataset(X_imbalanced, y_imbalanced)

# Calculate weights for each sample
class_sample_count = np.array([np.sum(y_imbalanced == t) for t in np.unique(y_imbalanced)])
weight = 1. / class_sample_count
samples_weight = np.array([weight[t] for t in y_imbalanced])
samples_weight = torch.from_numpy(samples_weight).float()

# Create weighted sampler
weighted_sampler = WeightedRandomSampler(
    weights=samples_weight,
    num_samples=len(samples_weight),
    replacement=True
)

# Create balanced DataLoader
balanced_loader = DataLoader(
    imbalanced_dataset, 
    batch_size=64, 
    sampler=weighted_sampler
)

# Check class distribution in a few batches
class_counts = {0: 0, 1: 0}
for i, (_, labels) in enumerate(balanced_loader):
    for label in labels:
        class_counts[label.item()] += 1
    if i >= 5:  # Check just a few batches
        break

print("Class distribution in weighted sampler batches:")
print(f"Class 0: {class_counts[0]} samples ({class_counts[0]/(class_counts[0]+class_counts[1])*100:.1f}%)")
print(f"Class 1: {class_counts[1]} samples ({class_counts[1]/(class_counts[0]+class_counts[1])*100:.1f}%)")

## 9. Working with Large Datasets

When datasets are too large to fit into memory, we need special techniques to process them efficiently.

### 9.1 Creating a Generator-based Dataset in PyTorch

In [ ]:
# Example: A generator-based dataset that loads data on-demand
class LazyDataset(Dataset):
    def __init__(self, data_source, transform=None):
        """
        data_source: Could be a list of file paths or other references
        transform: Optional transform to be applied on samples
        """
        self.data_source = data_source
        self.transform = transform
        
    def __len__(self):
        return len(self.data_source)
    
    def __getitem__(self, idx):
        """Load data only when needed"""
        # This is where you would load data from disk, database, etc.
        # For example: image_path = self.data_source[idx]
        #              image = Image.open(image_path)
        
        # For demonstration, we'll just generate random data
        sample = np.random.randn(3, 224, 224)  # Simulate RGB image
        label = np.random.randint(0, 10)  # Simulate class label
        
        if self.transform:
            sample = self.transform(sample)
            
        return sample, label

# Example of using LazyDataset
data_source = [f"dummy_file_{i}.jpg" for i in range(1000)]  # Just dummy paths
lazy_dataset = LazyDataset(data_source)

# Access a few items
for i in range(3):
    sample, label = lazy_dataset[i]
    print(f"Sample {i} shape: {sample.shape}, label: {label}")

### 9.2 Using TensorFlow's `tf.data.Dataset.from_generator`

In [ ]:
# Example: Create a generator function for TensorFlow
def data_generator():
    """Generate data on-the-fly"""
    for _ in range(1000):  # Simulating 1000 samples
        # Generate a random sample
        sample = np.random.randn(224, 224, 3).astype(np.float32)  # RGB image
        label = np.random.randint(0, 10)  # Class label
        yield sample, label

# Create a TensorFlow dataset from the generator
tf_lazy_dataset = tf.data.Dataset.from_generator(
    data_generator,
    output_signature=(
        tf.TensorSpec(shape=(224, 224, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
)

# Configure the dataset
tf_lazy_dataset = tf_lazy_dataset.batch(32).prefetch(tf.data.experimental.AUTOTUNE)

# Access a few batches
for i, (samples, labels) in enumerate(tf_lazy_dataset.take(3)):
    print(f"Batch {i} shape: {samples.shape}, labels shape: {labels.shape}")

### 9.3 Memory-Mapped Files with NumPy

In [ ]:
# Example: Using memory-mapped files for large arrays
import os
import tempfile
import shutil

# Create a temporary directory
temp_dir = tempfile.mkdtemp()

try:
    # Generate a large array
    data_shape = (10000, 1000)  # 10,000 samples with 1,000 features each
    mmap_filename = os.path.join(temp_dir, 'large_data.dat')
    
    # Create a memory-mapped array
    mmap_array = np.memmap(
        mmap_filename,
        dtype=np.float32,
        mode='w+',
        shape=data_shape
    )
    
    # Fill the array with random data (in chunks to save memory)
    chunk_size = 1000
    for i in range(0, data_shape[0], chunk_size):
        end = min(i + chunk_size, data_shape[0])
        mmap_array[i:end] = np.random.randn(end - i, data_shape[1])
    
    # Force data to be written to disk
    mmap_array.flush()
    
    # Now open the array in read-only mode
    mmap_array = np.memmap(
        mmap_filename,
        dtype=np.float32,
        mode='r',
        shape=data_shape
    )
    
    # Create a custom dataset that uses the memory-mapped array
    class MemmapDataset(Dataset):
        def __init__(self, memmap_data, labels=None, transform=None):
            self.data = memmap_data
            # Generate random labels if not provided
            self.labels = labels if labels is not None else np.random.randint(0, 2, size=len(memmap_data))
            self.transform = transform
            
        def __len__(self):
            return len(self.data)
        
        def __getitem__(self, idx):
            x = self.data[idx]
            y = self.labels[idx]
            
            if self.transform:
                x = self.transform(x)
                
            return x, y
    
    # Create dataset and dataloader
    memmap_dataset = MemmapDataset(mmap_array)
    memmap_loader = DataLoader(memmap_dataset, batch_size=64, shuffle=False)
    
    # Access a batch
    features, labels = next(iter(memmap_loader))
    print(f"Batch features shape: {features.shape}, labels shape: {labels.shape}")
    print(f"Features memory usage: {features.element_size() * features.nelement() / (1024 * 1024):.2f} MB")
    
finally:
    # Clean up temporary directory
    shutil.rmtree(temp_dir)

## 10. Creating Data Pipelines

Let's build complete data processing pipelines for both PyTorch and TensorFlow.

### 10.1 PyTorch Data Pipeline

In [ ]:
# Complete PyTorch Data Pipeline

class CompleteDataPipeline:
    def __init__(self, batch_size=32, num_workers=0, val_split=0.2):
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.val_split = val_split
        
    def get_transforms(self):
        """Define data transformations"""
        train_transform = transforms.Compose([
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        
        val_transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        
        return train_transform, val_transform
    
    def load_data(self, dataset_name='cifar10'):
        """Load a dataset"""
        train_transform, val_transform = self.get_transforms()
        
        if dataset_name.lower() == 'cifar10':
            # Load CIFAR-10 dataset
            train_dataset = torchvision.datasets.CIFAR10(
                root='./data', 
                train=True, 
                download=True, 
                transform=train_transform
            )
            
            test_dataset = torchvision.datasets.CIFAR10(
                root='./data', 
                train=False, 
                download=True, 
                transform=val_transform
            )
        else:
            raise ValueError(f"Dataset {dataset_name} not supported")
        
        return train_dataset, test_dataset
    
    def split_train_val(self, train_dataset):
        """Split training set into training and validation"""
        train_size = int((1 - self.val_split) * len(train_dataset))
        val_size = len(train_dataset) - train_size
        
        train_subset, val_subset = random_split(
            train_dataset, 
            [train_size, val_size],
            generator=torch.Generator().manual_seed(42)
        )
        
        return train_subset, val_subset
    
    def create_dataloaders(self, train_subset, val_subset, test_dataset=None):
        """Create DataLoaders for training, validation, and testing"""
        train_loader = DataLoader(
            train_subset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True
        )
        
        val_loader = DataLoader(
            val_subset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True
        )
        
        test_loader = None
        if test_dataset is not None:
            test_loader = DataLoader(
                test_dataset,
                batch_size=self.batch_size,
                shuffle=False,
                num_workers=self.num_workers,
                pin_memory=True
            )
        
        return train_loader, val_loader, test_loader
    
    def build_pipeline(self, dataset_name='cifar10'):
        """Build the complete data pipeline"""
        # Load datasets
        train_dataset, test_dataset = self.load_data(dataset_name)
        
        # Split training and validation
        train_subset, val_subset = self.split_train_val(train_dataset)
        
        # Create dataloaders
        train_loader, val_loader, test_loader = self.create_dataloaders(
            train_subset, val_subset, test_dataset
        )
        
        # Return loaders and dataset information
        dataset_info = {
            'name': dataset_name,
            'train_size': len(train_subset),
            'val_size': len(val_subset),
            'test_size': len(test_dataset) if test_dataset else None,
            'num_classes': len(train_dataset.classes) if hasattr(train_dataset, 'classes') else None,
            'class_names': train_dataset.classes if hasattr(train_dataset, 'classes') else None
        }
        
        return train_loader, val_loader, test_loader, dataset_info

# Create and use the PyTorch pipeline
pipeline = CompleteDataPipeline(batch_size=64, num_workers=0)
train_loader, val_loader, test_loader, dataset_info = pipeline.build_pipeline('cifar10')

print("PyTorch Data Pipeline Created:")
print(f"Dataset: {dataset_info['name']}")
print(f"Number of classes: {dataset_info['num_classes']}")
print(f"Class names: {dataset_info['class_names']}")
print(f"Training set size: {dataset_info['train_size']}")
print(f"Validation set size: {dataset_info['val_size']}")
print(f"Test set size: {dataset_info['test_size']}")

# Get a batch of data
images, labels = next(iter(train_loader))
print(f"\nBatch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Labels: {labels[:5]}")  # Print first 5 labels

# Visualize a few images
plt.figure(figsize=(10, 5))
for i in range(5):
    plt.subplot(1, 5, i+1)
    img = images[i].permute(1, 2, 0).numpy()
    # Denormalize the image
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    plt.title(dataset_info['class_names'][labels[i]])
    plt.axis('off')
plt.tight_layout()
plt.show()

### 10.2 TensorFlow Data Pipeline

In [ ]:
# Complete TensorFlow Data Pipeline

class TFDataPipeline:
    def __init__(self, batch_size=32, val_split=0.2, buffer_size=10000):
        self.batch_size = batch_size
        self.val_split = val_split
        self.buffer_size = buffer_size
        
    def preprocess_data(self, images, labels, is_training=True):
        """Apply preprocessing to images"""
        # Normalize pixel values
        images = tf.cast(images, tf.float32) / 255.0
        
        if is_training:
            # Data augmentation for training
            images = self._augment_images(images)
            
        return images, labels
    
    def _augment_images(self, images):
        """Apply data augmentation to images"""
        # Random flips
        images = tf.image.random_flip_left_right(images)
        
        # Random brightness and contrast adjustments
        images = tf.image.random_brightness(images, max_delta=0.1)
        images = tf.image.random_contrast(images, lower=0.9, upper=1.1)
        
        # Make sure pixel values are in [0, 1]
        images = tf.clip_by_value(images, 0.0, 1.0)
        
        return images
    
    def load_data(self, dataset_name='cifar10'):
        """Load a dataset"""
        if dataset_name.lower() == 'cifar10':
            # Load CIFAR-10 dataset
            (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
            # Flatten labels dimension
            y_train = tf.squeeze(y_train)
            y_test = tf.squeeze(y_test)
            # Get class names
            class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
                          'dog', 'frog', 'horse', 'ship', 'truck']
        elif dataset_name.lower() == 'mnist':
            # Load MNIST dataset
            (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
            # Add channel dimension to MNIST
            x_train = x_train[..., tf.newaxis]
            x_test = x_test[..., tf.newaxis]
            # Get class names
            class_names = [str(i) for i in range(10)]
        else:
            raise ValueError(f"Dataset {dataset_name} not supported")
        
        return (x_train, y_train), (x_test, y_test), class_names
    
    def create_datasets(self, x_train, y_train, x_test, y_test):
        """Create TensorFlow datasets with proper preprocessing"""
        # Calculate train/val split
        train_size = int((1 - self.val_split) * len(x_train))
        
        # Create training dataset
        train_dataset = tf.data.Dataset.from_tensor_slices(
            (x_train[:train_size], y_train[:train_size])
        )
        train_dataset = train_dataset.shuffle(self.buffer_size)
        train_dataset = train_dataset.map(
            lambda x, y: self.preprocess_data(x, y, is_training=True),
            num_parallel_calls=tf.data.experimental.AUTOTUNE
        )
        train_dataset = train_dataset.batch(self.batch_size)
        train_dataset = train_dataset.prefetch(tf.data.experimental.AUTOTUNE)
        
        # Create validation dataset
        val_dataset = tf.data.Dataset.from_tensor_slices(
            (x_train[train_size:], y_train[train_size:])
        )
        val_dataset = val_dataset.map(
            lambda x, y: self.preprocess_data(x, y, is_training=False),
            num_parallel_calls=tf.data.experimental.AUTOTUNE
        )
        val_dataset = val_dataset.batch(self.batch_size)
        val_dataset = val_dataset.prefetch(tf.data.experimental.AUTOTUNE)
        
        # Create test dataset
        test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test))
        test_dataset = test_dataset.map(
            lambda x, y: self.preprocess_data(x, y, is_training=False),
            num_parallel_calls=tf.data.experimental.AUTOTUNE
        )
        test_dataset = test_dataset.batch(self.batch_size)
        test_dataset = test_dataset.prefetch(tf.data.experimental.AUTOTUNE)
        
        return train_dataset, val_dataset, test_dataset
    
    def build_pipeline(self, dataset_name='cifar10'):
        """Build the complete data pipeline"""
        # Load data
        (x_train, y_train), (x_test, y_test), class_names = self.load_data(dataset_name)
        
        # Create TF datasets
        train_dataset, val_dataset, test_dataset = self.create_datasets(
            x_train, y_train, x_test, y_test
        )
        
        # Calculate split sizes
        train_size = int((1 - self.val_split) * len(x_train))
        val_size = len(x_train) - train_size
        test_size = len(x_test)
        
        # Return datasets and information
        dataset_info = {
            'name': dataset_name,
            'train_size': train_size,
            'val_size': val_size,
            'test_size': test_size,
            'num_classes': len(class_names),
            'class_names': class_names,
            'image_shape': x_train[0].shape
        }
        
        return train_dataset, val_dataset, test_dataset, dataset_info

# Create and use the TensorFlow pipeline
tf_pipeline = TFDataPipeline(batch_size=64)
train_ds, val_ds, test_ds, ds_info = tf_pipeline.build_pipeline('cifar10')

print("TensorFlow Data Pipeline Created:")
print(f"Dataset: {ds_info['name']}")
print(f"Number of classes: {ds_info['num_classes']}")
print(f"Class names: {ds_info['class_names']}")
print(f"Training set size: {ds_info['train_size']}")
print(f"Validation set size: {ds_info['val_size']}")
print(f"Test set size: {ds_info['test_size']}")
print(f"Image shape: {ds_info['image_shape']}")

# Get a batch of data
for images, labels in train_ds.take(1):
    print(f"\nBatch shape: {images.shape}")
    print(f"Labels shape: {labels.shape}")
    print(f"Labels: {labels[:5]}")  # Print first 5 labels
    
    # Visualize a few images
    plt.figure(figsize=(10, 5))
    for i in range(5):
        plt.subplot(1, 5, i+1)
        plt.imshow(images[i].numpy())
        plt.title(ds_info['class_names'][labels[i].numpy()])
        plt.axis('off')
    plt.tight_layout()
    plt.show()
    break

## 11. Optimizing Data Loading Performance

Let's explore techniques to optimize data loading performance.

### 11.1 Measuring Data Loading Speed

In [ ]:
import time

def measure_dataloader_speed(dataloader, num_batches=100):
    """Measure the speed of a dataloader in batches/second"""
    start_time = time.time()
    
    # Iterate through the specified number of batches
    for i, _ in enumerate(dataloader):
        if i >= num_batches - 1:
            break
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    
    batches_per_second = num_batches / elapsed_time
    return batches_per_second, elapsed_time

# Create a basic dataset and dataloader for testing
basic_dataset = torch.utils.data.TensorDataset(
    torch.randn(1000, 3, 224, 224),  # Simulating 1000 RGB images
    torch.randint(0, 10, (1000,))     # Class labels
)

# Create a basic dataloader with no optimizations
basic_loader = DataLoader(
    basic_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0  # Single process
)

# Measure speed (using fewer batches to avoid running too long)
batches_per_sec, elapsed_time = measure_dataloader_speed(basic_loader, num_batches=10)

print(f"Basic DataLoader speed: {batches_per_sec:.2f} batches/second")
print(f"Time to load 10 batches: {elapsed_time:.2f} seconds")

### 11.2 Optimizing PyTorch DataLoader Performance

In [ ]:
# Let's compare different configurations
def compare_dataloader_configs(dataset, batch_size=32, num_batches=10):
    """Compare different DataLoader configurations"""
    configs = {
        "basic": {"num_workers": 0, "pin_memory": False},
        "pin_memory": {"num_workers": 0, "pin_memory": True},
        "workers_2": {"num_workers": 2, "pin_memory": False},
        "workers_2_pin": {"num_workers": 2, "pin_memory": True},
        "workers_4_pin": {"num_workers": 4, "pin_memory": True}
    }
    
    results = {}
    
    for name, config in configs.items():
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=True,
            **config
        )
        
        # Warm up
        for _, _ in enumerate(loader):
            if _ >= 1:
                break
                
        # Measure
        try:
            batches_per_sec, elapsed = measure_dataloader_speed(loader, num_batches)
            results[name] = (batches_per_sec, elapsed, config)
        except Exception as e:
            results[name] = (0, 0, config)
            print(f"Error with config {name}: {e}")
    
    return results

# Use a smaller dataset for quicker testing
small_dataset = torch.utils.data.TensorDataset(
    torch.randn(200, 3, 224, 224),  # 200 RGB images
    torch.randint(0, 10, (200,))     # Class labels
)

# Compare configurations
try:
    results = compare_dataloader_configs(small_dataset)
    
    # Display results
    print("\nDataLoader Configuration Comparison:")
    print("-" * 80)
    print(f"{'Config Name':<15} {'Speed (batch/s)':<20} {'Time (s)':<15} {'Settings'}")
    print("-" * 80)
    
    for name, (speed, time, config) in results.items():
        print(f"{name:<15} {speed:<20.2f} {time:<15.2f} {config}")
except Exception as e:
    print(f"Error comparing configurations: {e}")
    print("This may be due to using a notebook environment that doesn't support multiprocessing.")

### 11.3 Optimizing TensorFlow Dataset Performance

In [ ]:
# Compare TensorFlow dataset optimizations
def measure_tf_dataset_speed(dataset, num_batches=100):
    """Measure the speed of a TensorFlow dataset in batches/second"""
    start_time = time.time()
    
    # Iterate through the specified number of batches
    for i, _ in enumerate(dataset):
        if i >= num_batches - 1:
            break
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    
    batches_per_second = num_batches / elapsed_time
    return batches_per_second, elapsed_time

def compare_tf_dataset_configs(batch_size=32, num_batches=10):
    """Compare different TensorFlow Dataset configurations"""
    # Create a dataset of random tensors
    features = tf.random.normal((1000, 224, 224, 3))
    labels = tf.random.uniform((1000,), maxval=10, dtype=tf.int32)
    
    configs = {
        "basic": lambda ds: ds.batch(batch_size),
        "shuffle": lambda ds: ds.shuffle(1000).batch(batch_size),
        "prefetch": lambda ds: ds.batch(batch_size).prefetch(tf.data.AUTOTUNE),
        "shuffle_prefetch": lambda ds: ds.shuffle(1000).batch(batch_size).prefetch(tf.data.AUTOTUNE),
        "cache_shuffle_prefetch": lambda ds: ds.cache().shuffle(1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    }
    
    results = {}
    
    for name, config_fn in configs.items():
        # Create base dataset
        base_ds = tf.data.Dataset.from_tensor_slices((features, labels))
        
        # Apply configuration
        dataset = config_fn(base_ds)
        
        # Warm up
        for _ in dataset.take(1):
            pass
                
        # Measure
        try:
            batches_per_sec, elapsed = measure_tf_dataset_speed(dataset, num_batches)
            results[name] = (batches_per_sec, elapsed)
        except Exception as e:
            results[name] = (0, 0)
            print(f"Error with config {name}: {e}")
    
    return results

# Compare TensorFlow dataset configurations
try:
    tf_results = compare_tf_dataset_configs()
    
    # Display results
    print("\nTensorFlow Dataset Configuration Comparison:")
    print("-" * 50)
    print(f"{'Config Name':<20} {'Speed (batch/s)':<15} {'Time (s)'}")
    print("-" * 50)
    
    for name, (speed, time) in tf_results.items():
        print(f"{name:<20} {speed:<15.2f} {time:.2f}")
except Exception as e:
    print(f"Error comparing TensorFlow configurations: {e}")

## Conclusion

In this notebook, we've covered essential concepts and techniques for working with datasets and data loaders in deep learning:

1. **Dataset Concepts**: Understanding training/validation/test splits, features and labels
2. **Custom Datasets**: Building custom dataset classes in PyTorch and TensorFlow
3. **Built-in Datasets**: Using pre-configured datasets like MNIST and CIFAR-10
4. **Data Loaders**: Implementing efficient batch processing with PyTorch DataLoader and TensorFlow Dataset API
5. **Data Preprocessing**: Applying normalization, standardization, and other transformations
6. **Data Augmentation**: Enhancing datasets with techniques like random crops, flips, and color adjustments
7. **Handling Imbalanced Datasets**: Using resampling and weighted sampling techniques
8. **Large Dataset Handling**: Working with datasets too large to fit in memory
9. **End-to-End Data Pipelines**: Creating complete data processing workflows
10. **Performance Optimization**: Techniques for faster data loading

Effective data handling is critical for successful deep learning projects. The techniques covered here will help you build efficient, scalable data pipelines for your models.